In [ ]:
import pandas as pd

Połączenie danych z MovieLens i z TMDB pobramych dla zbioru MovieLens

In [ ]:
movie_lens = pd.read_csv('../data/merged/movielens.csv')
movie_lens

In [ ]:
tmdb = pd.read_csv('../data/tmdb/tmdb_data_from_movieLens.csv')
tmdb

In [ ]:
df = pd.merge(
    tmdb,
    movie_lens.drop(columns=['movieId', 'title']),
    how='left',
    on='tmdbId').rename(columns={'top_5_tags': 'keywords'})
df['release_date'] = pd.to_datetime(df['release_date'])
df

In [ ]:
genres1 = df['genres_x'].str.lower().str.replace(' ', '')
genres2 = df['genres_y'].str.lower().str.replace('|', ',')
df['genres'] = (
    (genres1 + ',' + genres2)
    .str.split(',')
    .explode()
    .dropna()
    .str.strip()
    .loc[lambda x: x != '']
    .groupby(level=0)
    .unique()
    .apply(lambda x: ','.join(x))
)
df.drop(['genres_x', 'genres_y', 'imdbId'], axis=1, inplace=True)
df['genres']

Połączenie z dodatkowo pobranymi filmami z bazy TMDB, których nie było w MovieLens

In [ ]:
tmdb1 = pd.read_csv('../data/tmdb/tmdb_data.csv')
tmdb1['keywords'] = tmdb1['keywords'].str.replace(', ', ',')
tmdb1['genres'] = tmdb1['genres'].str.replace(', ', ',').str.lower()
tmdb1['release_date'] = pd.to_datetime(tmdb1['release_date'])
tmdb1['year'] = tmdb1['release_date'].dt.year
tmdb1

In [ ]:
df = pd.concat([df, tmdb1], ignore_index=True)
print(len(df))
df.drop_duplicates(inplace=True)
df

Połączenie danych o ludziach

In [ ]:
people = pd.read_csv('../data/tmdb/tmdb_people_movieLens.csv')
print(len(people))
people1 = pd.read_csv('../data/tmdb/tmdb_people.csv')
print(len(people1))
people = pd.concat([people, people1], ignore_index=True)
people

Do każdego filmu po jednym najpopularniejszym reżyserze, scenarzyście i producencie

In [ ]:
single_choice_departments = ['Production', 'Writing', 'Directing']
actors = people[people['known_for_department'] == 'Acting']
others = people[people['known_for_department'].isin(single_choice_departments)]
others_top = others.loc[others.groupby(['movie_id', 'known_for_department'])['popularity'].idxmax()]
people = pd.concat([actors, others_top], ignore_index=True)
people['known_for_department'] = people['known_for_department'].replace({
    'Acting': 'actor',
    'Writing': 'writer',
    'Directing': 'director',
    'Production': 'producer',
})
people.drop(columns=['job'], inplace=True)
people.rename(columns={'known_for_department': 'job', 'person_id': 'id'}, inplace=True)
people

Do każdego filmu 5 najpopularniejszych aktorów - piwot danych (1 to najbardziej popularny)

In [ ]:
actors = people[people['job'] == 'actor'].copy()
actors['rank'] = actors.groupby('movie_id')['popularity'].rank(method='first', ascending=False)
actors = actors[actors['rank'] <= 5]
actors['rank'] = actors['rank'].astype(int)

rows = []
for movie_id, group in actors.groupby("movie_id"):
    row = {"movie_id": movie_id}
    for i, person in group.iterrows():
        row[f"actor{person['rank']}_id"] = person['id']
        row[f"actor{person['rank']}_name"] = person['name']
        row[f"actor{person['rank']}_popularity"] = person['popularity']
        row[f"actor{person['rank']}_gender"] = person['gender']
    rows.append(row)
actors = pd.DataFrame(rows)
actors

Piwot pozostałych osób

In [ ]:
others = people[people['job'] != 'actor'].copy()
rows = []
for movie_id, group in others.groupby("movie_id"):
    row = {"movie_id": movie_id}
    for i, person in group.iterrows():
        row[f"{person['job']}_id"] = person['id']
        row[f"{person['job']}_name"] = person['name']
        row[f"{person['job']}_popularity"] = person['popularity']
        row[f"{person['job']}_gender"] = person['gender']
    rows.append(row)
others = pd.DataFrame(rows)
others

Połączenie danych o ludziach i filmach

In [ ]:
df = pd.merge(
    left=df,
    right=others,
    how='left',
    left_on='tmdbId',
    right_on='movie_id',
)
df = pd.merge(
    left=df,
    right=actors,
    how='left',
    left_on='tmdbId',
    right_on='movie_id',
)
df = df.drop(columns=['movie_id_x', 'movie_id_y'])
df.rename(columns={'top_5_tags': 'keywords'}, inplace=True)
df['keywords'] = df['keywords'].str.replace(', ', ',')
df

Uwzględnienie inflacji - dane są w dolarach amerykańskich, a więc uwzględniamy inflację tej waluty. Jako rok bazowy przyjmujemy 2025 rok i pod ten rok dostosowujemy wszystkie wartości.

In [ ]:
cpi = pd.read_excel('../data/cpi.xlsx', skiprows=11)
cpi = cpi[['Year', 'Annual']].set_index('Year').to_dict()['Annual']
cpi

In [ ]:
base_year = 2025
data = df.copy()
data['budget_adjusted'] = round(df['budget'] * (cpi[base_year] / data['year'].map(cpi)), 2)
data['revenue_adjusted'] = round(df['revenue'] * (cpi[base_year] / data['year'].map(cpi)), 2)
data

Usuwanie duplikatów tych samych tytułów, tmdbId i roku produkcji

In [ ]:
data['keywords_count'] = data['keywords'].fillna('').str.count(',')
data = data.sort_values(by='keywords_count', ascending=True)
data = data.drop(columns=['keywords_count'])

data = data.drop_duplicates(subset=['tmdbId', 'title', 'release_date'], keep='first')
data

In [ ]:
len(data[(data['revenue'] > 0)])

In [ ]:
data.to_csv('../data/merged/merged_data.csv')